# Pydantic 출력 검사 — 강사용 정답본

원본: `solution/Chapter01/02-langchain/03_output_parser.py`  
대응 슬라이드: `99_pdf/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf`

**학습 목표:** MovieReview 스키마 → 형식 지시 → 출력 파싱의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 영화 리뷰 생성을 위해 LLM을 호출한다. 모델이 스키마 형식을 어기면 파싱 오류가 날 수 있다.



## 흐름을 먼저 읽기

MovieReview 스키마 → 형식 지시 → 출력 파싱

핵심은 각 Runnable의 입력·출력과 반복 흐름이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: MovieReview 스키마 → 형식 지시 → 출력 파싱. 모델이 필드를 누락하면 어디서 오류가 나는가? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 영화 리뷰 생성을 위해 LLM을 호출한다. 모델이 스키마 형식을 어기면 파싱 오류가 날 수 있다.



In [ ]:
"""
Output Parser: LLM 출력을 구조화된 데이터로 변환하기

두 가지 방식 비교:
  1. PydanticOutputParser + format_instructions → 프롬프트에 형식 지시 포함
  2. .with_structured_output()                 → LLM의 function-calling 기능 활용 (권장)

실행:
  python 03_output_parser.py
"""

from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── 출력 스키마 정의 ───────────────────────────────────────────────────────────

class MovieReview(BaseModel):
    title: str = Field(description='영화 제목')
    genre: str = Field(description='장르 (예: 액션, 드라마, SF)')
    rating: float = Field(description='평점 (1.0 ~ 10.0)')
    summary: str = Field(description='100자 이내 한 줄 요약')
    recommend: bool = Field(description='추천 여부')


# ── 방법 1: PydanticOutputParser ──────────────────────────────────────────────
# 프롬프트에 JSON 형식 지시문을 주입하여 LLM이 구조화된 출력을 생성하도록 유도.
# LLM이 지시를 따르지 않으면 파싱 실패 가능.

print('=== 방법 1: PydanticOutputParser ===')

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_messages([
    ('system', '영화 리뷰 전문가야. 다음 형식으로 응답해.\n{format_instructions}'),
    ('human', '{movie} 리뷰 작성해줘'),
]).partial(format_instructions=parser.get_format_instructions())

chain1 = prompt | llm | parser
review1: MovieReview = chain1.invoke({'movie': '인터스텔라'})

print(f'제목   : {review1.title}')
print(f'장르   : {review1.genre}')
print(f'평점   : {review1.rating}')
print(f'요약   : {review1.summary}')
print(f'추천   : {"✓" if review1.recommend else "✗"}')


# ── 방법 2: .with_structured_output() ────────────────────────────────────────
# LLM의 function-calling(tool use) 기능을 활용하므로 더 안정적.
# 프롬프트에 형식 지시를 추가할 필요 없음.

print('\n=== 방법 2: with_structured_output() ===')

structured_llm = llm.with_structured_output(MovieReview)

prompt2 = ChatPromptTemplate.from_messages([
    ('system', '영화 리뷰 전문가야.'),
    ('human', '{movie} 리뷰 작성해줘'),
])

chain2 = prompt2 | structured_llm
review2: MovieReview = chain2.invoke({'movie': '기생충'})

print(f'제목   : {review2.title}')
print(f'장르   : {review2.genre}')
print(f'평점   : {review2.rating}')
print(f'요약   : {review2.summary}')
print(f'추천   : {"✓" if review2.recommend else "✗"}')


# ── 비교 요약 ─────────────────────────────────────────────────────────────────
print('''
┌─────────────────────┬──────────────────────────┬──────────────────────────┐
│                     │ PydanticOutputParser      │ with_structured_output() │
├─────────────────────┼──────────────────────────┼──────────────────────────┤
│ 안정성              │ LLM 지시 이행에 의존      │ function-calling 보장    │
│ 프롬프트 복잡도     │ format_instructions 필요  │ 프롬프트 단순            │
│ 구형 모델 지원      │ ✓ (모든 텍스트 LLM)       │ function-calling 필요    │
│ 권장               │ 구형 모델 / 커스텀 파싱    │ GPT-4o 이상 기본 권장    │
└─────────────────────┴──────────────────────────┴──────────────────────────┘
''')


## 결과를 검토하기

- 예상 동작: 형식에 맞으면 MovieReview 필드가 보이고 어기면 파싱 오류가 난다.
- 확인 질문: 모델이 필드를 누락하면 어디서 오류가 나는가?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
